In [2]:
import os, subprocess, time, json, random
import numpy as np, pandas as pd, torch, torch.nn as nn
from tqdm.auto import tqdm
from sklearn.metrics import roc_curve
from transformers import Wav2Vec2Model

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEV, MAXLEN = "cuda", 64600
print("GPU available:", torch.cuda.is_available())


BASE = "/kaggle/input/datasets/mohammedabdeldayem/avsspoof-2021"
METADATA = os.path.join(BASE, "DF-keys-full", "keys", "DF", "CM", "trial_metadata.txt")
DF_PARTS = [os.path.join(BASE, f"ASVspoof2021_DF_eval_part0{i}", "ASVspoof2021_DF_eval") for i in range(3)]
print("metadata exists:", os.path.exists(METADATA))
for p in DF_PARTS:
    print(p, "-> flac folder exists:", os.path.exists(os.path.join(p, "flac")))

df21 = pd.read_csv(METADATA, sep=r"\s+", header=None)[[1, 5]]
df21.columns = ["audio_id", "label"]
df21["label"] = df21["label"].map({"spoof": 0, "bonafide": 1})      # 0 = spoof, 1 = bona fide
lookup = {}
for part in DF_PARTS:
    d = os.path.join(part, "flac")
    for f in os.listdir(d):
        if f.endswith(".flac"):
            lookup[f[:-5]] = os.path.join(d, f)
df21["path"] = df21["audio_id"].map(lookup)
df21 = df21.dropna(subset=["path"]).reset_index(drop=True)
bona = df21[df21.label == 1]
spoof = df21[df21.label == 0].sample(n=len(bona), random_state=42)
df2021_balanced = pd.concat([spoof, bona], ignore_index=True).sample(frac=1, random_state=42).reset_index(drop=True)
print("test clips:", len(df2021_balanced), df2021_balanced.label.value_counts().to_dict())   # expect 33954, 16977 each

# ---- ASVspoof 2019 LA: TRAIN and DEV for the head ----
A19 = "/kaggle/input/datasets/awsaf49/asvpoof-2019-dataset/LA/LA"
CM = os.path.join(A19, "ASVspoof2019_LA_cm_protocols")
def load19(protocol, flac_dir):
    d = pd.read_csv(os.path.join(CM, protocol), sep=r"\s+", header=None)[[1, 4]]
    d.columns = ["audio_id", "label"]
    d["label"] = d["label"].map({"spoof": 0, "bonafide": 1})
    d["path"] = d["audio_id"].apply(lambda x: os.path.join(flac_dir, x + ".flac"))
    return d
df2019_train = load19("ASVspoof2019.LA.cm.train.trn.txt", os.path.join(A19, "ASVspoof2019_LA_train", "flac"))
df2019_dev = load19("ASVspoof2019.LA.cm.dev.trl.txt", os.path.join(A19, "ASVspoof2019_LA_dev", "flac"))
print("2019 train:", len(df2019_train), "| dev:", len(df2019_dev))
print("first train file exists:", os.path.exists(df2019_train.path.iloc[0]))
print("first dev file exists:", os.path.exists(df2019_dev.path.iloc[0]))

GPU available: True
metadata exists: True
/kaggle/input/datasets/mohammedabdeldayem/avsspoof-2021/ASVspoof2021_DF_eval_part00/ASVspoof2021_DF_eval -> flac folder exists: True
/kaggle/input/datasets/mohammedabdeldayem/avsspoof-2021/ASVspoof2021_DF_eval_part01/ASVspoof2021_DF_eval -> flac folder exists: True
/kaggle/input/datasets/mohammedabdeldayem/avsspoof-2021/ASVspoof2021_DF_eval_part02/ASVspoof2021_DF_eval -> flac folder exists: True
test clips: 33954 {1: 16977, 0: 16977}
2019 train: 25380 | dev: 24844
first train file exists: True
first dev file exists: True


In [4]:
def load_audio_ffmpeg(path):
    cmd = ["ffmpeg", "-v", "error", "-i", path, "-f", "f32le", "-ac", "1", "-ar", "16000", "-"]
    r = subprocess.run(cmd, stdout=subprocess.PIPE, stderr=subprocess.PIPE, check=True)
    return np.frombuffer(r.stdout, dtype=np.float32).copy()

w2v = Wav2Vec2Model.from_pretrained("facebook/wav2vec2-base").to(DEV).eval()
for p in w2v.parameters():
    p.requires_grad = False

def make_head():      # identical layout to AudioSpoofClassifier.classifier in model.py
    return nn.Sequential(nn.Linear(768, 128), nn.ReLU(), nn.Dropout(0.3), nn.Linear(128, 1))

def eer_percent(y_spoof, p_spoof):
    fpr, tpr, _ = roc_curve(y_spoof, p_spoof)
    fnr = 1 - tpr
    i = np.nanargmin(np.abs(fpr - fnr))
    return (fpr[i] + fnr[i]) / 2 * 100

@torch.no_grad()
def embed_paths(paths, desc=""):
    out = []
    for p in tqdm(list(paths), desc=desc):
        try:
            x = load_audio_ffmpeg(p)[:MAXLEN]
            x = (x - x.mean()) / np.sqrt(x.var() + 1e-7)
            x = torch.from_numpy(x).float().unsqueeze(0).to(DEV)
            out.append(w2v(x).last_hidden_state.mean(dim=1).squeeze(0).cpu())
        except Exception:
            out.append(None)
    return out

def stack_ok(vecs, labels):
    keep = [i for i, v in enumerate(vecs) if v is not None]
    print(f"decoded {len(keep)}/{len(vecs)} clips")
    return torch.stack([vecs[i] for i in keep]), np.asarray(labels)[keep]

def balanced(df, n):
    a = df[df.label == 0].sample(n, random_state=SEED)
    b = df[df.label == 1].sample(n, random_state=SEED)
    return pd.concat([a, b]).sample(frac=1, random_state=SEED).reset_index(drop=True)
print("ready")

pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  380MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

model.safetensors: reconstructing file:   0%|          |  0.00B /  380MB            

Loading weights:   0%|          | 0/211 [00:00<?, ?it/s]

model.safetensors: downloading bytes:           |  0.00B            

[transformers] Wav2Vec2Model LOAD REPORT from: facebook/wav2vec2-base
Key                          | Status     |  | 
-----------------------------+------------+--+-
quantizer.codevectors        | UNEXPECTED |  | 
project_hid.bias             | UNEXPECTED |  | 
quantizer.weight_proj.weight | UNEXPECTED |  | 
project_q.bias               | UNEXPECTED |  | 
project_hid.weight           | UNEXPECTED |  | 
quantizer.weight_proj.bias   | UNEXPECTED |  | 
project_q.weight             | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


ready


In [5]:
tracker = None
try:
    subprocess.run(["pip", "install", "-q", "codecarbon"], check=True)
    from codecarbon import EmissionsTracker
    tracker = EmissionsTracker(project_name="wav2vec2_head", log_level="error", save_to_file=False)
    tracker.start()
except Exception as e:
    print("CodeCarbon not available (fine, energy just won't be reported):", e)

tr_df, dv_df = balanced(df2019_train, 2500), balanced(df2019_dev, 1000)
t0 = time.time()
Xtr, ytr = stack_ok(embed_paths(tr_df.path, "train"), 1 - tr_df.label.values)   # spoof = 1 from here on
Xdv, ydv = stack_ok(embed_paths(dv_df.path, "dev"), 1 - dv_df.label.values)
t_embed_trdev = time.time() - t0
print(f"embedding time (train+dev): {t_embed_trdev:.0f}s")

[codecarbon WARNING @ 00:12:38] Multiple instances of codecarbon are allowed to run at the same time.


train:   0%|          | 0/5000 [00:00<?, ?it/s]

decoded 5000/5000 clips


dev:   0%|          | 0/2000 [00:00<?, ?it/s]

decoded 2000/2000 clips
embedding time (train+dev): 1629s


In [6]:
head = make_head().to(DEV)
opt = torch.optim.Adam(head.parameters(), lr=1e-3, weight_decay=1e-4)
lossf = nn.BCEWithLogitsLoss()
Xtr_g, ytr_g, Xdv_g = Xtr.to(DEV), torch.tensor(ytr, dtype=torch.float32, device=DEV), Xdv.to(DEV)

best_eer, best_state, history = 100.0, None, []
t0 = time.time()
for ep in range(40):
    head.train()
    perm = torch.randperm(len(Xtr_g), device=DEV)
    total = 0.0
    for i in range(0, len(perm), 64):
        idx = perm[i:i + 64]
        opt.zero_grad()
        loss = lossf(head(Xtr_g[idx]).squeeze(-1), ytr_g[idx])
        loss.backward(); opt.step()
        total += loss.item() * len(idx)
    head.eval()
    with torch.no_grad():
        s = torch.sigmoid(head(Xdv_g).squeeze(-1)).cpu().numpy()
    e = eer_percent(ydv, s)
    history.append((ep + 1, total / len(perm), float(e)))
    if e < best_eer:
        best_eer, best_state = e, {k: v.clone() for k, v in head.state_dict().items()}
    print(f"epoch {ep+1:2d}  loss {total/len(perm):.4f}  dev EER {e:.2f}%")
t_train = time.time() - t0
head.load_state_dict(best_state)
torch.save(head.state_dict(), "/kaggle/working/head.pt")
print(f"\nbest dev EER {best_eer:.2f}% | training {t_train:.1f}s | saved /kaggle/working/head.pt")

epoch  1  loss 0.4780  dev EER 11.15%
epoch  2  loss 0.2722  dev EER 9.10%
epoch  3  loss 0.2069  dev EER 7.80%
epoch  4  loss 0.1704  dev EER 6.65%
epoch  5  loss 0.1498  dev EER 6.10%
epoch  6  loss 0.1288  dev EER 5.40%
epoch  7  loss 0.1173  dev EER 5.30%
epoch  8  loss 0.1056  dev EER 4.65%
epoch  9  loss 0.0964  dev EER 4.70%
epoch 10  loss 0.0915  dev EER 4.25%
epoch 11  loss 0.0833  dev EER 4.50%
epoch 12  loss 0.0765  dev EER 4.40%
epoch 13  loss 0.0722  dev EER 4.20%
epoch 14  loss 0.0689  dev EER 4.05%
epoch 15  loss 0.0656  dev EER 4.15%
epoch 16  loss 0.0611  dev EER 4.10%
epoch 17  loss 0.0603  dev EER 3.90%
epoch 18  loss 0.0557  dev EER 3.90%
epoch 19  loss 0.0582  dev EER 3.90%
epoch 20  loss 0.0609  dev EER 3.55%
epoch 21  loss 0.0473  dev EER 3.25%
epoch 22  loss 0.0460  dev EER 3.10%
epoch 23  loss 0.0474  dev EER 3.75%
epoch 24  loss 0.0473  dev EER 3.00%
epoch 25  loss 0.0473  dev EER 2.95%
epoch 26  loss 0.0413  dev EER 2.90%
epoch 27  loss 0.0477  dev EER 2.90%


In [7]:


t0 = time.time()
Xte, yte = stack_ok(embed_paths(df2021_balanced.path, "test"), 1 - df2021_balanced.label.values)
t_embed_test = time.time() - t0
head.eval()
with torch.no_grad():
    te = torch.sigmoid(head(Xte.to(DEV)).squeeze(-1)).cpu().numpy()

test_eer = eer_percent(yte, te)
pred = (te >= 0.5).astype(int)
acc = (pred == yte).mean() * 100
tn = int(((pred == 0) & (yte == 0)).sum()); fp = int(((pred == 1) & (yte == 0)).sum())
fn = int(((pred == 0) & (yte == 1)).sum()); tp = int(((pred == 1) & (yte == 1)).sum())
print(f"TEST EER (wav2vec2 head): {test_eer:.2f}%    [AASIST baseline: 19.49%]")
print(f"accuracy at 0.5: {acc:.2f}%")
print(f"bona fide kept: {tn} | bona fide flagged as spoof: {fp} | spoof missed: {fn} | spoof caught: {tp}")
emissions_kg = None
if tracker is not None:
    try: emissions_kg = tracker.stop()
    except Exception as e: print("CodeCarbon stop failed:", e)
print("estimated emissions, kg CO2e:", emissions_kg)

test:   0%|          | 0/33954 [00:00<?, ?it/s]

decoded 33954/33954 clips
TEST EER (wav2vec2 head): 17.15%    [AASIST baseline: 19.49%]
accuracy at 0.5: 79.10%
bona fide kept: 10533 | bona fide flagged as spoof: 6444 | spoof missed: 652 | spoof caught: 16325
estimated emissions, kg CO2e: 0.03539729193197896


In [2]:
import os, subprocess, importlib.util, torch

if not os.path.exists("/kaggle/working/aasist"):
    subprocess.run(["git", "clone", "https://github.com/clovaai/aasist.git",
                    "/kaggle/working/aasist"], check=True)

if "model" not in globals():
    spec = importlib.util.spec_from_file_location("AASIST", "/kaggle/working/aasist/models/AASIST.py")
    aasist = importlib.util.module_from_spec(spec); spec.loader.exec_module(aasist)
    cfg = {"architecture": "AASIST", "nb_samp": 64600, "first_conv": 128,
           "filts": [70, [1,32], [32,32], [32,64], [64,64]], "gat_dims": [64,32],
           "pool_ratios": [0.5,0.7,0.5,0.5], "temperatures": [2.0,2.0,100.0,100.0]}
    model = aasist.Model(cfg)

n_aasist = sum(p.numel() for p in model.parameters())
print(f"AASIST total parameters: {n_aasist:,}")
print("wav2vec2 head trainable parameters: 98,561")

Cloning into '/kaggle/working/aasist'...


AASIST total parameters: 297,866
wav2vec2 head trainable parameters: 98,561
